# DSE - Data Mining
> Name: Pratyush Pravanjan Pradhan<br>
> Roll no: 251248<br>
> Course: B.Sc. Computer Science hons.<br>
> Project: Breast Cancer (Wisconsin) - preprocessing, classification, clustering and association analysis<br>

## Aim
To apply the data mining techniques covered in the syllabus on the **Breast Cancer Wisconsin (Diagnostic)** dataset and find out which cell measurements separate a **malignant** tumour from a **benign** one.

## Objectives
1. Clean and preprocess the data (aggregation, discretization, binarization, sampling, standardization, normalization).
2. **Association analysis** - find rules like `radius_High -> Malignant` using support and confidence.
3. **Clustering** - apply K-means (without using the diagnosis) and check if it finds the two groups.
4. **Classification** - predict malignant / benign using the K-Nearest Neighbours algorithm.

## Importing the dataset
The dataset is available inside `scikit-learn`, so no download or upload is needed.<br>
It has 569 patients. Each row is a breast lump sample, and the values are measured from the cell nuclei in a microscope image.<br>
For every measurement the dataset has 3 versions (`mean`, `error`, `worst`). **We will only use the `mean` columns**, so we get 10 features.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer

In [ ]:
data = load_breast_cancer(as_frame=True)
full = data.frame

# keeping only the 'mean' columns
mean_cols = [c for c in full.columns if c.startswith('mean ')]
df = full[mean_cols].copy()

# removing the word 'mean ' from the column names to keep them short
df.columns = [c.replace('mean ', '').replace(' ', '_') for c in mean_cols]

# in sklearn 0 = malignant and 1 = benign, so we flip it to make malignant = 1
df['diagnosis'] = (full['target'] == 0).astype(int)
df['diagnosis_name'] = df['diagnosis'].map({1: 'Malignant', 0: 'Benign'})

print(df.shape)
print(df.head())

## Dataset Description
| Column | Meaning |
|---|---|
| `radius` | average distance from centre of the nucleus to its edge |
| `texture` | variation in gray-scale values of the pixels |
| `perimeter` | length of the boundary of the nucleus |
| `area` | area of the nucleus |
| `smoothness` | local variation in the radius lengths |
| `compactness` | perimeter² / area - 1 |
| `concavity` | how deep the dents in the boundary are |
| `concave_points` | number of dents in the boundary |
| `symmetry` | symmetry of the nucleus shape |
| `fractal_dimension` | "coastline" complexity of the boundary |
| `diagnosis` | **target** - 1 = Malignant, 0 = Benign |

In [ ]:
print(df.info())
print(df.describe())

In [ ]:
print(df['diagnosis_name'].value_counts())

## We will now clean and process it

> **Why are we doing this?**<br>
> Real data can have missing values or repeated rows, and these give wrong results in every later step (averages, distances, clusters). So we check the quality of the data first. In this dataset nothing was missing, so no filling or removing was needed.

In [ ]:
#checking if any column has missing values
print(df.isnull().sum())
print("duplicate rows:", df.duplicated().sum())

No column has missing values and there are no duplicate rows, so no filling or removing is needed.

## Aggregation.

> **Why are we doing this?**<br>
> Aggregation summarises many rows into a few numbers (mean, min, max) for each group. Here we group by diagnosis to compare the two types of tumour side by side. It gives a first idea of which features differ between malignant and benign, before using any algorithm.

In [ ]:
aggregation = df.groupby('diagnosis_name').agg({
    'radius': 'mean',
    'texture': 'mean',
    'area': 'mean',
    'concavity': 'mean'
})

print(aggregation)

In [ ]:
aggregation = df.groupby('diagnosis_name').agg({
    'radius': ['mean', 'min', 'max'],
    'area': ['mean', 'median'],
    'concavity': ['mean', 'max']
})

print(aggregation)

Malignant tumours have a clearly higher average radius, area and concavity than benign ones.

# Discretization

> **Why are we doing this?**<br>
Discretization converts a continuous number (like radius = 17.99) into a few categories (Small / Medium / Large or Low / Medium / High).<br>
- `pd.cut` makes bins with the cut-points **we choose**.<br>
- `pd.qcut` makes bins with **equal number of rows** in each.<br>
We need this because association rules work on **items** (like `radius_High`), not on continuous numbers.

In [ ]:
# custom bins on radius
radius_bins = [0, 12, 15, 40]
radius_labels = ['Small', 'Medium', 'Large']

df['radius_category'] = pd.cut(
    df['radius'],
    bins=radius_bins,
    labels=radius_labels
)

print(df[['radius', 'radius_category']].head())
print(df['radius_category'].value_counts())

In [ ]:
# equal-frequency bins (each bin gets nearly the same number of rows) using qcut
levels = ['Low', 'Medium', 'High']
for col in ['radius', 'texture', 'smoothness', 'concavity']:
    df[col + '_level'] = pd.qcut(df[col], q=3, labels=levels)

print(df[['radius', 'radius_level', 'concavity', 'concavity_level']].head())
print(df['concavity_level'].value_counts())

# Binarization

> **Why are we doing this?**<br>
> Binarization turns a value into **0 or 1** using a threshold (here, the median). It is the simplest form of discretization: "is the radius large or not?". The cross-table afterwards shows how strongly this one yes/no flag is related to the diagnosis.

In [ ]:
df['large_radius'] = (df['radius'] >= df['radius'].median()).astype(int)
df['high_concavity'] = (df['concavity'] >= df['concavity'].median()).astype(int)
print(df[['radius', 'concavity', 'large_radius', 'high_concavity']].head())

In [ ]:
# how many malignant / benign in each group
print(pd.crosstab(df['large_radius'], df['diagnosis_name']))

# Sampling
selecting a subset of the entire dataset.

> **Why are we doing this?**<br>
> Sampling takes a smaller subset of the data. It is used when the full data is too big to process quickly, or to test an idea fast. A good sample should keep the same proportions as the full data, which is why we compare the percentage of malignant in the sample and in the full data. `random_state=42` makes the same sample come every time.

In [ ]:
sample = df.sample(
    n=100,
    random_state=42
)

print(sample[['radius', 'texture', 'area', 'diagnosis_name']].head(10))
print("\nsample size:", len(sample))

In [ ]:
# checking if the sample has the same ratio of malignant as the full data
print("full data  :", round(df['diagnosis'].mean() * 100, 1), "% malignant")
print("sample     :", round(sample['diagnosis'].mean() * 100, 1), "% malignant")

## Standardization
$$z = \frac{x - \mu}{\sigma}$$

> **Why are we doing this?**<br>
> Our columns are on very different scales (`area` is in hundreds, `smoothness` is about 0.1). Standardization shifts every column to **mean 0 and standard deviation 1**, so that every feature is on the same scale. It is used before K-means because K-means works on distances.

In [ ]:
# selecting the 10 numerical feature columns
features = ['radius', 'texture', 'perimeter', 'area', 'smoothness',
            'compactness', 'concavity', 'concave_points', 'symmetry', 'fractal_dimension']

df_standardized = df.copy()
df_standardized[features] = (df[features] - df[features].mean()) / df[features].std()
print(df_standardized[features].head())

In [ ]:
print(df_standardized[features].mean().round(3))
print(df_standardized[features].std().round(3))

## Normalization
$$x' = \frac{x - x_{min}}{x_{max} - x_{min}}$$

> **Why are we doing this?**<br>
> Normalization (min-max) squeezes every column into the range **0 to 1**. Like standardization, it stops big-valued columns from dominating the distance. We use it before KNN. The difference: standardization keeps the shape around the mean, while min-max keeps all values between 0 and 1.

In [ ]:
df_normalized = df.copy()
df_normalized[features] = (df[features] - df[features].min()) / (df[features].max() - df[features].min())
print(df_normalized[features].head())
print("\nmin:", df_normalized[features].min().min(), " max:", df_normalized[features].max().max())

`area` is in hundreds/thousands while `smoothness` is around 0.1, so without scaling `area` would dominate any distance calculation. That is why scaling is needed before KNN and K-means.

# Association Analysis
Association rules need **transactions made of items**, so we use the discretized columns. Every patient becomes one transaction, for example:

`{radius_High, texture_High, concavity_High, Malignant}`

## Making the transactions

> **Why are we doing this?**<br>
> Association analysis finds **"if-then" patterns** in the data, like *if radius is High then the tumour is Malignant*. It is mostly known from market-basket analysis (people who buy bread also buy butter). Here every patient is a "basket" and the items are the discretized features and the diagnosis. Its purpose is to get **simple, human-readable rules** about which feature levels go with which diagnosis.

In [ ]:
transactions = []
for i in range(len(df)):
    t = {
        'radius_' + str(df['radius_level'][i]),
        'texture_' + str(df['texture_level'][i]),
        'smoothness_' + str(df['smoothness_level'][i]),
        'concavity_' + str(df['concavity_level'][i]),
        df['diagnosis_name'][i]
    }
    transactions.append(t)

print("Total transactions:", len(transactions))
for t in transactions[:5]:
    print(sorted(t))

## Support for the items in itemset
$$Support(X)=\frac{\text{Number of transactions containing X}}{\text{Total number of transaction}}$$
We will store it in a dataframe to display it better in a tabular form.

In [ ]:
itemsets = [
    {'Malignant'},
    {'Benign'},
    {'radius_High'},
    {'radius_Low'},
    {'concavity_High'},
    {'radius_High', 'concavity_High'},
    {'radius_High', 'Malignant'},
    {'radius_Low', 'Benign'},
    {'radius_High', 'concavity_High', 'Malignant'}
]

In [ ]:
total_transac=len(transactions)
support_itemsets=[]
for i in itemsets:
    count = sum(i.issubset(tran) for tran in transactions)
    support = count/total_transac
    support_itemsets.append({
        'items':', '.join(sorted(i)),
        'count':count,
        'support':round(support,4)
    })
print(pd.DataFrame(support_itemsets).to_string())

## Support and Confidence for association rules
We will make two separate function for each. </br>
For a rule $X \rightarrow Y$:
$$
Support(X \rightarrow Y) = Support(X \cup Y)
$$
$$
Confidence(X \rightarrow Y) = \frac{Support(X \cup Y)}{Support(X)}
$$

> **How to read the numbers**<br>
> - **Support** = how common the pattern is in the whole data. A rule with very low support might be just a coincidence.<br>
> - **Confidence** = out of the patients who have the left side (X), how many also have the right side (Y). It tells how reliable the rule is.<br>
>
> Example: confidence of `radius_High -> Malignant` = 0.88 means 88% of the patients with a High radius were malignant.<br>
> We only trust a rule when **both** support and confidence are above the minimum values we set.

In [ ]:
def support(transaction,X,Y=None):
    if Y is not None:
        return support(transaction,X.union(Y))
    count=sum(X.issubset(j) for j in transaction)
    return count/len(transaction)
def confidence(transaction,X,Y):
    return support(transaction,X,Y)/support(transaction,X)

And again store it in a dataframe to display it

In [ ]:
rules = [
    ({'radius_High'}, {'Malignant'}),
    ({'concavity_High'}, {'Malignant'}),
    ({'radius_High', 'concavity_High'}, {'Malignant'}),
    ({'radius_Low'}, {'Benign'}),
    ({'concavity_Low'}, {'Benign'}),
    ({'radius_Low', 'concavity_Low'}, {'Benign'}),
    ({'Malignant'}, {'radius_High'}),
    ({'texture_High'}, {'Malignant'}),
    ({'smoothness_High'}, {'Malignant'})
]

In [ ]:
results = []
for X, Y in rules:
    results.append({
        "Antecedent": ", ".join(sorted(X)),
        "Consequent": ", ".join(sorted(Y)),
        "Support": round(support(transactions, X, Y), 4),
        "Confidence": round(confidence(transactions, X, Y), 4)
    })
df_rules = pd.DataFrame(results)
print(df_rules.to_string(index=False))

### Strong rules
A rule is considered strong if it has a good **support** and a high **confidence**. Using minimum support = 0.15 and minimum confidence = 0.80:

In [ ]:
min_support = 0.15
min_confidence = 0.80

strong = df_rules[(df_rules['Support'] >= min_support) & (df_rules['Confidence'] >= min_confidence)]
print(strong.to_string(index=False))

# K-means Clustering
Clustering is **unsupervised** - we will not give the diagnosis to the algorithm. Then we compare the clusters with the real diagnosis.

The Euclidean distance between a data point `x` and centroid `c` is:

$$
d(x,c) = \sqrt{\sum_j (x_j-c_j)^2}
$$

The MSE used here is the mean of the squared distances of each point from its assigned centroid:

$$
MSE = \frac{1}{n}\sum_{i=1}^{n} d(x_i,c_i)^2
$$

> **Why are we doing this?**<br>
> Clustering is **unsupervised** - it groups similar records together without being told the answer. Here we hide the diagnosis and ask: *does the data separate itself into natural groups?* If it does, then the measurements really contain the information that distinguishes the tumours.<br>
>
> **How K-means works:**<br>
> 1. Pick `k` random points as centroids.<br>
> 2. Assign every point to its nearest centroid.<br>
> 3. Move each centroid to the mean of its points.<br>
> 4. Repeat 2 and 3 until the centroids stop moving.<br>
>
> The MSE goes down in every iteration because points keep getting closer to their centroids. We use **standardized** data here, since K-means works on distance.

In [ ]:
def kmeans(data, k, max_iter=100):
  data=np.asarray(data,dtype=float)
  idx = np.random.choice(len(data), k, replace=False)
  centroids=data[idx].copy()
  mse_history=[]

  for i in range(max_iter):
    dist = np.linalg.norm(
      data[:, np.newaxis, :] - centroids,
      axis=2
    )
    labels=np.argmin(dist,axis=1)
    new_centroids=np.array([np.mean(data[labels==j],axis=0) if np.any(labels==j) else centroids[j] for j in range(k)])

    point_dist=np.min(dist,axis=1)
    mse=np.mean(point_dist**2)
    mse_history.append(mse)

    if np.all(centroids==new_centroids):
      break
    centroids=new_centroids
  return centroids,mse_history,labels

In [ ]:
# using the standardized features (all 10 columns, diagnosis is NOT used)
cluster_data = df_standardized[features].values
print(cluster_data.shape)

In [ ]:
np.random.seed(1)
k = 2

centroids, mse_history, labels = kmeans(cluster_data, k=k, max_iter=100)

print("Number of iterations:", len(mse_history))
print("Final MSE:", mse_history[-1])
print("Cluster sizes:", np.bincount(labels))

In [ ]:
plt.figure(figsize=(7, 5))

plt.plot(
    range(1, len(mse_history) + 1),
    mse_history,
    marker="o"
)

plt.xlabel("Iteration")
plt.ylabel("MSE")
plt.title("MSE vs Iteration (k=2)")
plt.xticks(range(1, len(mse_history) + 1))
plt.grid()
plt.show()

print("MSE after each iteration:")
for i, mse in enumerate(mse_history, start=1):
    print(f"Iteration {i}: {mse:.4f}")

In [ ]:
# Visualize the clusters using two features (radius and texture, standardized)
plt.figure(figsize=(7, 5))

for cluster in range(k):
    pts = cluster_data[labels == cluster]
    plt.scatter(pts[:, 0], pts[:, 1], label=f"Cluster {cluster + 1}")

plt.scatter(
    centroids[:, 0],
    centroids[:, 1],
    marker="X",
    s=200,
    c="black",
    label="Centroids"
)

plt.xlabel("radius (standardized)")
plt.ylabel("texture (standardized)")
plt.title("K-Means Clustering (k=2)")
plt.legend()
plt.grid()
plt.show()

### Comparing the clusters with the real diagnosis

> **Why are we doing this?**<br>
> K-means never saw the diagnosis, so we use it only now to **check** the result. We count how many malignant and benign patients fell into each cluster. If each cluster is mostly one type, the clustering found a real pattern. Purity is the percentage of patients that are in the cluster of their own type.

In [ ]:
print(pd.crosstab(labels, df['diagnosis_name'], rownames=['cluster']))

# each cluster is named after the majority diagnosis inside it
table = pd.crosstab(labels, df['diagnosis'])
correct = table.max(axis=1).sum()
print("\nCluster purity:", round(correct / len(df) * 100, 2), "%")

Even though K-means never saw the diagnosis, the two clusters match the real malignant / benign groups quite well.

In [ ]:
# comparing different values of k
results = []
for k in [2, 3, 4, 5]:
    np.random.seed(1)
    c_k, mse_k, labels_k = kmeans(cluster_data, k=k, max_iter=100)
    results.append({
        "k": k,
        "iterations": len(mse_k),
        "final_MSE": mse_k[-1]
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

In [ ]:
for k in [2, 3, 4, 5]:
    np.random.seed(1)
    c_k, mse_k, labels_k = kmeans(cluster_data, k=k, max_iter=100)
    plt.plot(range(1, len(mse_k) + 1), mse_k, marker="o", label=f"k={k}")

plt.xlabel("Iteration")
plt.ylabel("MSE")
plt.title("MSE vs Iteration for Different Values of k")
plt.legend()
plt.grid()
plt.show()

> **Why the elbow graph?**<br>
> K-means needs `k` to be chosen in advance. MSE always goes down when k increases (with k = number of points it would be 0), so the lowest MSE is not the best k. Instead we look for the **elbow**, the point after which increasing k gives only a small improvement.

In [ ]:
# elbow graph: final MSE for k = 1 to 8
final_mse = []
for k in range(1, 9):
    np.random.seed(1)
    _, mse_k, _ = kmeans(cluster_data, k=k, max_iter=100)
    final_mse.append(mse_k[-1])

plt.plot(range(1, 9), final_mse, marker="o")
plt.xlabel("k")
plt.ylabel("Final MSE")
plt.title("Elbow graph")
plt.grid()
plt.show()

# Classification - K-Nearest Neighbours (KNN)
**Idea:** to classify a new patient, find the `k` closest patients in the training data (Euclidean distance) and take the majority vote of their diagnosis.

$$d(x,y) = \sqrt{\sum_j (x_j-y_j)^2}$$

**Steps:** split into train/test &rarr; normalize &rarr; apply KNN &rarr; find accuracy.

> **Why are we doing this?**<br>
> Classification is **supervised** - the model learns from patients whose diagnosis is already known, and then predicts the diagnosis of a new patient. KNN is based on a simple idea: **similar patients have the same diagnosis**. So for a new patient, it looks at the `k` closest patients and takes a majority vote.

> **Why do we split into train and test?**<br>
> If we test the model on the same data it learned from, the accuracy looks great but means nothing. So we keep 20% of the data hidden as a **test set** to see how the model performs on patients it has never seen.

In [ ]:
np.random.seed(42)

X = df[features].values
y = df['diagnosis'].values

# shuffling and splitting 80% train, 20% test
idx = np.random.permutation(len(X))
split = int(0.8 * len(X))
train_idx, test_idx = idx[:split], idx[split:]

X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]
print("train:", X_train.shape, " test:", X_test.shape)

> **Why normalize using only the training data?**<br>
> The test set represents "new, unseen" patients, so its values should not influence anything the model uses. We calculate min and max from the training data only, and apply the same values to the test data.

In [ ]:
# normalizing using the min and max of the TRAINING data only
mn = X_train.min(axis=0)
mx = X_train.max(axis=0)
X_train_n = (X_train - mn) / (mx - mn)
X_test_n = (X_test - mn) / (mx - mn)

In [ ]:
def knn_predict(X_train, y_train, X_test, k):
    predictions = []
    for point in X_test:
        dist = np.sqrt(np.sum((X_train - point) ** 2, axis=1))   # distance to every training point
        nearest = np.argsort(dist)[:k]                           # indices of k closest
        votes = np.bincount(y_train[nearest])                    # count 0s and 1s
        predictions.append(np.argmax(votes))                     # majority vote
    return np.array(predictions)

In [ ]:
k = 5
y_pred = knn_predict(X_train_n, y_train, X_test_n, k)

accuracy = np.mean(y_pred == y_test)
print(f"Accuracy for k={k}: {accuracy:.4f}")

> **Why a confusion matrix and not just accuracy?**<br>
> Accuracy hides the type of mistakes. The confusion matrix shows them:<br>
> - **TP** - malignant, predicted malignant (correct)<br>
> - **TN** - benign, predicted benign (correct)<br>
> - **FP** - benign, predicted malignant (false alarm)<br>
> - **FN** - malignant, predicted benign (**dangerous**, a missed cancer)<br>
>
> Because FN is the most harmful here, we look at **recall** along with accuracy.

In [ ]:
# confusion matrix
TP = np.sum((y_pred == 1) & (y_test == 1))
TN = np.sum((y_pred == 0) & (y_test == 0))
FP = np.sum((y_pred == 1) & (y_test == 0))
FN = np.sum((y_pred == 0) & (y_test == 1))

print(pd.DataFrame(
    [[TN, FP], [FN, TP]],
    index=['Actual Benign', 'Actual Malignant'],
    columns=['Predicted Benign', 'Predicted Malignant']
))

precision = TP / (TP + FP)
recall = TP / (TP + FN)
print(f"\nPrecision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")

$$Precision = \frac{TP}{TP+FP} \qquad Recall = \frac{TP}{TP+FN}$$
**Recall** is the important one here, because a malignant tumour wrongly predicted as benign (FN) is the most dangerous mistake.

> **Why try different values of k?**<br>
> A very small k (like 1) copies the single nearest point and is affected by noise. A very large k averages too many points and ignores local patterns. So we test several values and choose one that gives stable, good accuracy.

In [ ]:
# trying different values of k
results = []
for k in [1, 3, 5, 7, 9, 11]:
    pred = knn_predict(X_train_n, y_train, X_test_n, k)
    results.append({"k": k, "accuracy": np.mean(pred == y_test)})
print(pd.DataFrame(results).to_string(index=False))

> **Why compare with and without normalization?**<br>
> This proves that scaling is really needed. Without it, `area` (hundreds) dominates the distance and the other features are ignored, so the accuracy drops.

In [ ]:
# effect of normalization: same KNN on raw (unscaled) data
pred_raw = knn_predict(X_train, y_train, X_test, 5)
print("Accuracy WITHOUT normalization:", round(np.mean(pred_raw == y_test), 4))
print("Accuracy WITH normalization   :", round(np.mean(y_pred == y_test), 4))

# Conclusion
- The data had no missing values. Preprocessing techniques (aggregation, discretization, binarization, sampling, standardization, normalization) were applied on the 10 `mean` features.
- **Association rules** showed that large radius / high concavity cells are strongly associated with *Malignant*, and small radius / low concavity with *Benign*.
- **K-means** with k = 2, without using the diagnosis, produced two clusters that match the real diagnosis closely.
- **KNN** classified malignant / benign tumours with high accuracy, and normalization improved its accuracy because the features were on very different scales.